# PDF Parser Experiments

This notebook is used to experiment with PDF parsing and text extraction quality.

## Purpose

- Compare pdfplumber extraction vs OCR output on the same documents
- Identify common noise patterns in government tender PDFs
- Tune the `TextCleaner` configuration
- Spot-check that clause numbers are preserved after cleaning

## Setup

Place test PDFs in `data/raw-tenders/` before running this notebook.
Results are not committed — this is a scratch notebook.

In [ ]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

from src.parser.cleaner import TextCleaner
from src.parser.pdf_parser import PDFParser

# List available test PDFs
raw_dir = Path("../data/raw-tenders")
pdfs = list(raw_dir.glob("*.pdf"))
print(f"Found {len(pdfs)} PDF(s):")
for p in pdfs:
    print(f"  {p.name} ({p.stat().st_size / 1024:.0f} KB)")

## Extract and Clean a Single PDF

In [ ]:
# Change this to the PDF you want to inspect
pdf_path = pdfs[0] if pdfs else None

if pdf_path:
    parser = PDFParser()
    raw_text = parser.extract_text(pdf_path)
    cleaner = TextCleaner()
    clean_text = cleaner.clean(raw_text)

    print(f"Raw length:   {len(raw_text):,} chars")
    print(f"Clean length: {len(clean_text):,} chars")
    print(f"Reduction:    {(1 - len(clean_text) / len(raw_text)) * 100:.1f}%")
    print()
    print("--- First 2000 chars of clean text ---")
    print(clean_text[:2000])
else:
    print("No PDFs found. Add PDFs to data/raw-tenders/")

## Clause Extraction Check

In [ ]:
from src.extractor.clause_extractor import ClauseExtractor

if pdf_path:
    extractor = ClauseExtractor()
    clauses = extractor.extract_clauses(clean_text)
    print(f"Found {len(clauses)} clauses")
    print()
    for c in clauses[:10]:
        print(f"  [{c['clause_number']}] {c['heading'][:60]}  ({len(c['text'])} chars)")